# Task 5: Custom Transformer Backpropagation & Multi-Head Gradient Tracking Engine

## Mathematical Derivatives
$$\frac{\partial L}{\partial V} = A^T \frac{\partial L}{\partial O}, \quad \frac{\partial L}{\partial Q} = \frac{1}{\sqrt{d_k}} \frac{\partial L}{\partial S} K, \quad \frac{\partial L}{\partial K} = \frac{1}{\sqrt{d_k}} \left(\frac{\partial L}{\partial S}\right)^T Q$$


In [1]:
import numpy as np

# Analytical backpropagation function calculating partial derivatives for Query, Key, and Value
def manual_attention_backward(Q, K, V, dL_dO):
    d_k = Q.shape[-1]
    scores = np.matmul(Q, K.T) / np.sqrt(d_k)
    A = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    A /= np.sum(A, axis=-1, keepdims=True)
    
    dL_dV = np.matmul(A.T, dL_dO)
    dL_dA = np.matmul(dL_dO, V.T)
    dL_dS = A * (dL_dA - np.sum(dL_dA * A, axis=-1, keepdims=True))
    
    dL_dQ = np.matmul(dL_dS, K) / np.sqrt(d_k)
    dL_dK = np.matmul(dL_dS.T, Q) / np.sqrt(d_k)
    return dL_dQ, dL_dK, dL_dV


In [2]:
# Track gradient magnitudes across simulated sequence lengths N in [16, 64, 256]
seq_lengths = [16, 64, 256]
head_dim = 16

print(f"{'Seq Length N':<15} | {'dQ Norm':<12} | {'dK Norm':<12} | {'dV Norm':<12}")
print("-" * 58)

np.random.seed(42)
for N in seq_lengths:
    Q = np.random.randn(N, head_dim)
    K = np.random.randn(N, head_dim)
    V = np.random.randn(N, head_dim)
    dL_dO = np.random.randn(N, head_dim)
    
    dQ, dK, dV = manual_attention_backward(Q, K, V, dL_dO)
    print(f"{N:<15} | {np.linalg.norm(dQ):<12.4f} | {np.linalg.norm(dK):<12.4f} | {np.linalg.norm(dV):<12.4f}")


Seq Length N    | dQ Norm      | dK Norm      | dV Norm     
----------------------------------------------------------
16              | 3.6920       | 3.9268       | 5.6679      
64              | 6.1489       | 5.8993       | 7.5411      
256             | 6.1466       | 6.9906       | 6.3799      
